# Notebook 3 — Fairness Evaluation & Visualisation
## Derm Fairness Project · Fitzpatrick17k

This notebook loads all 5 trained model variants, runs them on the **held-out test set**,
and computes the full fairness metric suite:

- False Negative Rate (FNR) per group — *primary metric* (missing a malignant case = harmful)
- Recall / Sensitivity per group
- Worst-group recall
- Balanced accuracy
- Macro recall
- Demographic parity gap (recall gap between best and worst group)

All metrics are displayed in a single comparison table and a grouped bar chart.

---
### MIMIC portability
Only the data loading changes. The entire `evaluate_model()` function, metric table, and chart code
are **100% reusable** for MIMIC — just swap group names from skin tones to ethnicity/language.

## 0 · Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from pathlib import Path
from sklearn.metrics import (
    balanced_accuracy_score, recall_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
GROUPS = ['light', 'medium', 'dark']

## 1 · Reload test set & model architecture

In [ ]:
# Re-import Dataset class from Notebook 2
# (In practice, move shared classes to a utils.py in your repo)
import requests
from io import BytesIO
from PIL import Image
from torch.utils.data import Dataset

IMG_DIR = Path('data/images')

def get_image(url, cache_dir=IMG_DIR):
    img_id = url.split('/')[-1].split('?')[0][:64]
    local_path = cache_dir / f"{img_id}.jpg"
    if local_path.exists():
        return Image.open(local_path).convert('RGB')
    try:
        resp = requests.get(url, timeout=10)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        img.save(local_path)
        return img
    except Exception:
        return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=128)


class DermDataset(Dataset):
    def __init__(self, df, transform=None, cache_dir=IMG_DIR):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = get_image(row['url'], self.cache_dir)
        if self.transform:
            img = self.transform(img)
        return img, int(row['binary_label']), str(row['group'])


def build_model():
    m = models.efficientnet_b0(weights=None)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, 1))
    return m.to(DEVICE)


eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

df_test = pd.read_csv('data/test.csv')
test_dataset = DermDataset(df_test, eval_transform)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
print(f'Test set: {len(df_test)} samples')

## 2 · Inference helper

In [ ]:
def get_predictions(model, loader):
    model.eval()
    probs, labels, groups = [], [], []
    with torch.no_grad():
        for imgs, lbs, grps in loader:
            logits = model(imgs.to(DEVICE)).squeeze(1).cpu()
            probs.extend(torch.sigmoid(logits).numpy())
            labels.extend(lbs.numpy())
            groups.extend(grps)
    return np.array(probs), np.array(labels), np.array(groups)

## 3 · Core fairness metric function

Single function that takes predictions + optional per-group thresholds and returns
a dict of every metric we care about.

In [ ]:
def compute_fairness_metrics(
    probs: np.ndarray,
    labels: np.ndarray,
    groups: np.ndarray,
    group_thresholds: dict = None,
    default_threshold: float = 0.5
) -> dict:
    """
    Compute per-group and overall fairness metrics.

    Args:
        probs: predicted probabilities (N,)
        labels: ground truth binary labels (N,)
        groups: group membership strings (N,)
        group_thresholds: optional dict {group: threshold} for Mitigation 5
        default_threshold: used if group not in group_thresholds

    Returns:
        dict with per-group metrics + overall metrics
    """
    if group_thresholds is None:
        group_thresholds = {}

    # Apply per-group thresholds
    preds = np.array([
        int(p >= group_thresholds.get(g, default_threshold))
        for p, g in zip(probs, groups)
    ])

    metrics = {}

    # --- Per-group metrics ---
    group_recalls = []
    for grp in GROUPS:
        mask = groups == grp
        if mask.sum() == 0:
            continue
        g_preds  = preds[mask]
        g_labels = labels[mask]

        tp  = ((g_preds == 1) & (g_labels == 1)).sum()
        fn  = ((g_preds == 0) & (g_labels == 1)).sum()
        fp  = ((g_preds == 1) & (g_labels == 0)).sum()
        tn  = ((g_preds == 0) & (g_labels == 0)).sum()

        recall  = tp / (tp + fn + 1e-8)
        fnr     = fn / (tp + fn + 1e-8)
        bal_acc = balanced_accuracy_score(g_labels, g_preds)

        metrics[f'{grp}_recall']  = round(recall,  4)
        metrics[f'{grp}_fnr']     = round(fnr,     4)
        metrics[f'{grp}_bal_acc'] = round(bal_acc, 4)
        metrics[f'{grp}_n']       = int(mask.sum())
        group_recalls.append(recall)

    # --- Overall metrics ---
    metrics['worst_group_recall']   = round(min(group_recalls), 4)
    metrics['macro_recall']         = round(np.mean(group_recalls), 4)
    metrics['recall_gap']           = round(max(group_recalls) - min(group_recalls), 4)
    metrics['overall_bal_acc']      = round(balanced_accuracy_score(labels, preds), 4)
    metrics['overall_recall']       = round(recall_score(labels, preds, zero_division=0), 4)

    return metrics

## 4 · Evaluate all 5 models

In [ ]:
with open('data/group_thresholds.json') as f:
    group_thresholds = json.load(f)

model_configs = [
    ('Baseline',           'data/model_baseline.pt',   None),
    ('Balanced Sampling',  'data/model_balanced.pt',   None),
    ('Loss Reweighting',   'data/model_reweighted.pt', None),
    ('Targeted Finetune',  'data/model_targeted.pt',   None),
    ('Threshold Tuning',   'data/model_baseline.pt',   group_thresholds),  # same model, per-group thresholds
]

all_results = {}

for name, weights_path, thresholds in model_configs:
    print(f'Evaluating: {name} ...')
    model = build_model()
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    probs, labels, groups = get_predictions(model, test_loader)
    metrics = compute_fairness_metrics(probs, labels, groups, group_thresholds=thresholds)
    all_results[name] = metrics
    print(f"  Worst-group recall: {metrics['worst_group_recall']} | Macro recall: {metrics['macro_recall']} | Recall gap: {metrics['recall_gap']}")

print('\nDone!')

## 5 · Summary table

In [ ]:
rows = []
for name, m in all_results.items():
    rows.append({
        'Model': name,
        'Light Recall': m.get('light_recall', '-'),
        'Medium Recall': m.get('medium_recall', '-'),
        'Dark Recall': m.get('dark_recall', '-'),
        'Light FNR': m.get('light_fnr', '-'),
        'Medium FNR': m.get('medium_fnr', '-'),
        'Dark FNR': m.get('dark_fnr', '-'),
        'Worst-Group Recall': m['worst_group_recall'],
        'Macro Recall': m['macro_recall'],
        'Recall Gap': m['recall_gap'],
        'Overall Bal Acc': m['overall_bal_acc'],
    })

results_df = pd.DataFrame(rows).set_index('Model')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
results_df

In [ ]:
results_df.to_csv('data/fairness_results.csv')
print('Saved data/fairness_results.csv')

## 6 · Main results chart — Recall by skin tone group across all mitigations

This is the **figure for your paper/presentation**. Each cluster is one mitigation;
bars are coloured by skin tone group.

In [ ]:
model_names = list(all_results.keys())
x = np.arange(len(model_names))
width = 0.25

group_colors = {'light': '#f4c59f', 'medium': '#c68642', 'dark': '#4a2912'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Recall by group
ax = axes[0]
for i, grp in enumerate(GROUPS):
    vals = [all_results[name].get(f'{grp}_recall', 0) for name in model_names]
    ax.bar(x + (i - 1) * width, vals, width, label=f'{grp} skin',
           color=group_colors[grp], edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Recall (malignant cases)')
ax.set_title('Malignant Case Recall by Skin Tone Group')
ax.set_ylim(0, 1.05)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, label='80% recall target')
ax.legend(fontsize=9)

# (b) False Negative Rate by group
ax2 = axes[1]
for i, grp in enumerate(GROUPS):
    vals = [all_results[name].get(f'{grp}_fnr', 0) for name in model_names]
    ax2.bar(x + (i - 1) * width, vals, width, label=f'{grp} skin',
            color=group_colors[grp], edgecolor='white', linewidth=0.5)

ax2.set_xticks(x)
ax2.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('False Negative Rate (missed malignant cases)')
ax2.set_title('FNR by Skin Tone Group (lower = better)')
ax2.set_ylim(0, 1.05)
ax2.axhline(0.2, color='gray', linestyle='--', linewidth=0.8, label='20% FNR threshold')
ax2.legend(fontsize=9)

plt.suptitle('Fairness Across Mitigations — Fitzpatrick17k Dermatology', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fairness_chart.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved data/fairness_chart.png')

## 7 · Worst-group recall & recall gap summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

worst_recalls = [all_results[n]['worst_group_recall'] for n in model_names]
recall_gaps   = [all_results[n]['recall_gap'] for n in model_names]

axes[0].bar(model_names, worst_recalls, color='#4c72b0')
axes[0].set_title('Worst-Group Recall (higher = better)')
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
axes[0].axhline(0.8, color='gray', linestyle='--', linewidth=0.8)

axes[1].bar(model_names, recall_gaps, color='#dd8452')
axes[1].set_title('Recall Gap (lower = more equitable)')
axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('data/worst_group_gap.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved data/worst_group_gap.png')

## 8 · Print final narrative summary

In [ ]:
print('=' * 60)
print('FAIRNESS EVALUATION SUMMARY')
print('=' * 60)

baseline = all_results['Baseline']
print(f"\nBaseline worst-group recall: {baseline['worst_group_recall']}")
print(f"Baseline recall gap (light vs dark): {baseline['recall_gap']}")
print()

for name, m in all_results.items():
    delta_worst = m['worst_group_recall'] - baseline['worst_group_recall']
    delta_gap   = baseline['recall_gap'] - m['recall_gap']  # positive = improved equity
    print(f"{name:25s} | worst-group Δ: {delta_worst:+.4f} | gap Δ: {delta_gap:+.4f}")

print()
best_model = max(all_results.items(), key=lambda x: x[1]['worst_group_recall'])[0]
print(f"Best mitigation by worst-group recall: {best_model}")

---
### ✅ Notebook 3 complete

**Outputs:**
- `data/fairness_results.csv` — full metric table
- `data/fairness_chart.png` — recall & FNR by group across all mitigations
- `data/worst_group_gap.png` — worst-group recall & equity gap

---
### Porting to MIMIC
When MIMIC data is available:
1. Replace `DermDataset` with `MIMICNotesDataset` (tokenizes discharge summaries with ClinicalBERT)
2. Replace `build_model()` with ClinicalBERT + linear head
3. Change `GROUPS = ['light','medium','dark']` to `GROUPS = ['white','black','hispanic','asian']`
4. Change `binary_label` to `hospital_expire_flag`
5. `compute_fairness_metrics()` — **no changes needed**
6. All charts — **no changes needed**